# Bird counts: 10-day comparison, Location A vs Location B

Unique flying birds detected per recording day, from the batch results in `batch_runs/`.

- **09-30_A and 10-09_B use the 1080p recount.** Their 4K results (603k, 760k) were implausible.
- **Hatched = only counted at 4K** (09-29_B, 10-05_B, 10-06_A). These may not be directly comparable to 1080p days.
- Location A has no video for 10-07, so it has 9 days.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm

HERE = Path.cwd()          # run from batch_runs/
LOGS = HERE / "logs"
RECOUNT_1080 = {"2026-09-30_A", "2026-10-09_B"}   # 4K result replaced by the 1080p recount

COLOR = {"A": "#2a78d6", "B": "#eb6834"}
INK, MUTED, GRID, AXIS = "#0b0b0b", "#898781", "#e1e0d9", "#c3c2b7"
BLUES = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#2a78d6", "#1c5cab", "#104281", "#0d366b"]
plt.rcParams.update({
    "font.family": ["Segoe UI", "DejaVu Sans"], "font.size": 10,
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": AXIS, "axes.labelcolor": INK, "axes.titleweight": "bold", "axes.titlesize": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK, "ytick.labelcolor": INK,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8,
})


def final_result(path):
    """The detector's final result dict from a heartbeat log."""
    result = None
    for line in open(path):
        try:
            e = json.loads(line)
        except json.JSONDecodeError:
            continue
        if e.get("event") == "result":
            result = e["result"]
    return result


days, per_minute = [], {}
for r in pd.read_csv(HERE / "results.csv").itertuples():
    key = f"{r.date}_{r.location}"
    res = final_result(LOGS / f"{key}_1080.jsonl" if key in RECOUNT_1080 else LOGS / f"{key}.jsonl")
    if res is None:
        continue
    # The detector bins by int(fps) frames (29 for 29.97 footage), so one of its
    # "minutes" is 60*29/29.97 s of real time; place each bin at its true clock time.
    minute_s = 60 * res["fps"] / r.fps
    counts = np.array(json.loads(res["birds_per_minute"]))
    start = pd.Timestamp(r.recording_start)
    per_minute[key] = pd.Series(counts, index=start + pd.to_timedelta(np.arange(len(counts)) * minute_s, unit="s"))
    hours = res["frames_processed"] / r.fps / 3600
    days.append({
        "date": pd.Timestamp(r.date), "location": r.location,
        "start": start, "end": start + pd.Timedelta(hours=hours), "hours": hours,
        "unique_birds": res["unique_flying_birds"],
        "birds_per_hour": res["unique_flying_birds"] / hours,
        "peak_in_view": res["max_concurrent_birds"],
        "source": "1080p recount" if key in RECOUNT_1080 else ("4K" if r.height > 1080 else "1080p"),
    })
days = pd.DataFrame(days).sort_values(["location", "date"], ignore_index=True)
all_dates = sorted(days["date"].unique())
print(f"{len(days)} recordings loaded: "
      + ", ".join(f"{loc}: {n} days" for loc, n in days["location"].value_counts().sort_index().items()))

## 1. Summary per day

In [ ]:
for loc in ["A", "B"]:
    t = days[days.location == loc]
    table = pd.DataFrame({
        "Date": t.date.dt.strftime("%a %b %d"),
        "Recorded": t.start.dt.strftime("%H:%M") + "–" + t.end.dt.strftime("%H:%M"),
        "Hours": t.hours.map("{:.1f}".format),
        "Unique birds": t.unique_birds.map("{:,}".format),
        "Birds / hour": t.birds_per_hour.map("{:,.0f}".format),
        "Peak in view": t.peak_in_view,
        "Counted at": t.source,
    })
    total = t.unique_birds.sum()
    print(f"\nLocation {loc}: {len(t)} days, {total:,} birds in total, "
          f"busiest {t.loc[t.unique_birds.idxmax(), 'date']:%b %d}, quietest {t.loc[t.unique_birds.idxmin(), 'date']:%b %d}")
    print(table.to_string(index=False))

## 2. Daily totals

Same y-axis for both locations. Hours of footage are under each date, since a shorter recording means fewer birds.

In [ ]:
def k(n):
    return f"{n / 1000:.0f}k" if n >= 10_000 else f"{n / 1000:.1f}k".replace(".0k", "k") if n >= 1000 else f"{n:,.0f}"


fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
x = np.arange(len(all_dates))
for ax, loc in zip(axes, ["A", "B"]):
    t = days[days.location == loc].set_index("date").reindex(all_dates)
    four_k = (t.source == "4K").to_numpy()
    vals = t.unique_birds.fillna(0).to_numpy()
    ax.bar(x[~four_k], vals[~four_k], width=0.7, color=COLOR[loc], zorder=2)
    ax.bar(x[four_k], vals[four_k], width=0.7, color=COLOR[loc] + "40", edgecolor=COLOR[loc],
           hatch="///", linewidth=0, zorder=2, label="counted at 4K only")
    for xi, v, missing in zip(x, vals, t.unique_birds.isna()):
        ax.annotate("no video" if missing else k(v), (xi, v), ha="center", va="bottom",
                    fontsize=9, color=MUTED if missing else INK, xytext=(0, 3), textcoords="offset points")
    ax.set_xticks(x, [f"{d:%b %d}\n" + ("–" if np.isnan(h) else f"{h:.1f} h")
                      for d, h in zip(all_dates, t.hours)], fontsize=9)
    ax.set_title(f"Location {loc}", color=COLOR[loc], loc="left")
    ax.yaxis.set_major_formatter(lambda v, _: k(v) if v else "0")
    ax.tick_params(length=0)
    if four_k.any():
        ax.legend(frameon=False, loc="upper right", fontsize=9)
axes[0].set_ylabel("Unique flying birds per day")
fig.tight_layout()
plt.show()

## 3. When are birds active?

New birds per 30 minutes by clock time. Darker = more birds; log scale, so quiet days still show.
Grey = not recorded.

In [ ]:
BIN = "30min"
slots = pd.timedelta_range("07:00:00", "20:00:00", freq=BIN, closed="left")

def day_matrix(loc):
    """Rows = all_dates, columns = clock-time slots; NaN where nothing was recorded."""
    m = np.full((len(all_dates), len(slots)), np.nan)
    for i, d in enumerate(all_dates):
        key = f"{d:%Y-%m-%d}_{loc}"
        if key not in per_minute:
            continue
        s = per_minute[key]
        binned = s.groupby((s.index - s.index.normalize()).floor(BIN)).sum()
        for j, slot in enumerate(slots):
            if slot in binned.index:
                m[i, j] = binned[slot]
    return m

mats = {loc: day_matrix(loc) for loc in ["A", "B"]}
cmap = LinearSegmentedColormap.from_list("blues", BLUES)
cmap.set_bad("#f0efec")      # not recorded
cmap.set_under(BLUES[0])     # recorded, zero birds
vmax = np.nanmax([np.nanmax(m) for m in mats.values()])
norm = LogNorm(vmin=1, vmax=vmax)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for ax, loc in zip(axes, ["A", "B"]):
    m = mats[loc].copy()
    m[m == 0] = 0.5          # zero -> "under" colour, kept distinct from not-recorded
    im = ax.imshow(np.ma.masked_invalid(m), aspect="auto", cmap=cmap, norm=norm, interpolation="nearest")
    src = days[days.location == loc].set_index("date").source
    ax.set_yticks(range(len(all_dates)),
                  [f"{d:%b %d}" + ("  (4K)" if src.get(d) == "4K" else "") for d in all_dates], fontsize=9)
    ax.set_title(f"Location {loc}", color=COLOR[loc], loc="left")
    ax.grid(False)
    ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)
hours = [j for j, s in enumerate(slots) if s.seconds % 3600 == 0]
axes[-1].set_xticks([h - 0.5 for h in hours], [f"{slots[h].seconds // 3600:02d}:00" for h in hours])
axes[-1].set_xlabel("Time of day")
cb = fig.colorbar(im, ax=axes, shrink=0.6, pad=0.015)
cb.set_label("New birds per 30 min")
cb.ax.yaxis.set_major_formatter(lambda v, _: k(v))
cb.outline.set_visible(False)
plt.show()

## Notes

- **What is counted:** *unique flying birds* is the number of distinct tracks the detector confirmed as flying birds. A bird that leaves the frame and comes back is counted again, so this measures **activity**, not how many individual birds there are.
- **4K vs 1080p:** the two 4K days we rechecked at 1080p dropped about 5× (603k → 112k, 760k → 146k). The detector's size limits are in pixels, so the remaining hatched 4K days may be over-counted too. Recount them at 1080p before drawing firm conclusions from them.
- **Unequal recordings:** 10-16_A is only 6.9 h (ends 14:35), 09-29_A ends 16:44, and both 10-20 recordings start at 09:11. Use *Birds / hour* or the heatmap for fair comparisons.
- **Clock times** are corrected for the detector's rounded frame rate (its "minute" is really 58 s), so they are true times of day.